<a href="https://colab.research.google.com/github/Fespinoza1477/walmart-hurricane-project/blob/main/2026_10_23_midterm_checkpoint_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Midterm Checkpoint

**Lab — 2026-10-23 · Fall 2026**  

---

## Lab 09 — Midterm checkpoint, week 1 of 3

**What you submit: the link to your team's main repo. Only that.** One submission per team. Do not upload separate files.

Put this notebook in the repo (the `notebooks/` folder is fine) and push it, so everything we check is in one place.

**You do not have to fill this all out.** It is a checkpoint, not a finished project. Skip any section you have not started and write "not started" next to it. What we must see is:

1. **Your charter**, committed to the repo
2. **Any progress** you have made, with code that ran and output you can point at

Start with the team setup, then work down the same eight things we listed at Monday's kickoff, as far as you have gotten. This is graded on evidence, not on intentions.

The purpose is to catch a stalled team while there is still time to unstall it. If something is not working, say so here plainly. A checkpoint that honestly reports "the API join is not working and here is the error" is worth more than one that claims progress that does not exist, and it gets you help on Monday.

### Team setup

This part is required. Fill it in once for the whole team.

You need:

- Your team name, and every member with a GitHub handle (2 to 4 people)
- A link to your team's **main repo**. This is the one link you submit.
- A link to `CHARTER.md` in that repo, the page on GitHub, not a file on someone's laptop

Nice to have, and also in your charter: this week's notebook owner, your meeting time, and the five worst problems in the data from the profile you ran on Monday. Be specific: "`unit_price` is stored as text and some values start with `$`" beats "prices are messy."

In [ ]:
team_setup = {
    'team_name': 'The Three Musketeers',
    'members': [                                  # everyone on the team, 2 to 4 people
        {'name': 'Fabrizzio Espinoza', 'github': 'Fespinoza1477'},
        {'name': 'Micheal Hunter', 'github': 'michealhunter12'},
    ],
    'repo_url': 'https://github.com/Fespinoza1477/walmart-hurricane-project$0',                           # your main team repo: https://github.com/...
    'charter_url': 'https://github.com/Fespinoza1477/walmart-hurricane-project/blob/60a2f2310f2bed83c1f4aedb13d72048b0a88184/CHARTER.md',                        # link to CHARTER.md in that repo
    'notebook_owner_this_week': 'Everyone',           # nice to have
    'meeting_time': 'Whenever needed',                       # nice to have, e.g. 'Tue and Thu 7pm'
    'five_worst_problems': ['Determining what to keep or delete', 'communication being bad (not a problem so far)', 'People not doing their share of their work(not a problem so far)', 'Not meeting frequently', 'determining how to approach certain questions.'],              # nice to have, five specific problems
}

def check_team(t, quiet=False):
    def real(v):
        return isinstance(v, str) and v.strip() != '' and not v.startswith('TODO')
    required = {
        'team name': real(t['team_name']),
        '2 to 4 members, each with a name and GitHub handle':
            2 <= len(t['members']) <= 4 and all(real(m['name']) and real(m['github']) for m in t['members']),
        'main repo link': real(t['repo_url']) and t['repo_url'].startswith('https://github.com/'),
        'charter link points at CHARTER.md': real(t['charter_url']) and t['charter_url'].endswith('CHARTER.md'),
    }
    nice = {
        'notebook owner': real(t['notebook_owner_this_week']),
        'meeting time': real(t['meeting_time']),
        'five specific problems': len(t['five_worst_problems']) >= 5 and all(real(p) for p in t['five_worst_problems']),
    }
    if not quiet:
        print('REQUIRED')
        for label, good in required.items():
            print('  PASS' if good else '  FAIL', '-', label)
        print('NICE TO HAVE')
        for label, good in nice.items():
            print('  done' if good else '  not yet', '-', label)
        print()
        print('team setup complete:', all(required.values()))
    return all(required.values())

check_team(team_setup)

REQUIRED
  PASS - team name
  PASS - 2 to 4 members, each with a name and GitHub handle
  PASS - main repo link
  PASS - charter link points at CHARTER.md
NICE TO HAVE
  done - notebook owner
  done - meeting time
  done - five specific problems

team setup complete: True


True

Now show that the charter is actually committed. This is required. Run the next cell and leave its output in the notebook. Point `REPO_PATH` at your local clone of the team repo. If it prints a line like `a1b2c3d Add team charter`, you are done. If it prints "no commits found," the charter is not in the repo yet, so commit it and push before you submit.

In [ ]:
import subprocess

REPO_PATH = 'Fespinoza1477/walmart-hurricane-project'      # your local clone of the team repo

result = subprocess.run(['git', '-C', REPO_PATH, 'log', '--oneline', '--', 'CHARTER.md'],
                        capture_output=True, text=True)
print(result.stdout.strip() or 'no commits found for CHARTER.md')

no commits found for CHARTER.md


### 1 — Your cleaning pipeline as functions

**TODO:** paste the cleaning code you have written so far. It should be functions, not loose cells, and it should cover at minimum: duplicates, `unit_price` to numeric (it comes in as text, some with a `$`), the negative `quantity` values with your decision applied, `timestamp` parsed, and `store_id` cleaned up so it will join to the store file.

Print the row count before and after. We want to see how many rows your decisions removed.

In [ ]:
# TODO
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import sqlite3 as sql
import datetime as dt
import matplotlib.dates as mdates
import plotly.express as px
import plotly.graph_objects as go
import re
import pandas as pd, numpy as np, requests, time
# CSV Files
store_locations = 'https://raw.githubusercontent.com/Fespinoza1477/walmart-hurricane-project/main/Data/store_locations.csv'
transactions = 'https://raw.githubusercontent.com/Fespinoza1477/walmart-hurricane-project/main/Data/walmart_transactions.csv'
distribution_center = 'https://raw.githubusercontent.com/Fespinoza1477/walmart-hurricane-project/main/Data/distribution_centers.csv'
product_catalog = 'https://raw.githubusercontent.com/Fespinoza1477/walmart-hurricane-project/main/Data/product_catalog.csv'
# Database File
!wget -O walmart.db "https://raw.githubusercontent.com/Fespinoza1477/walmart-hurricane-project/main/Data/inventory_and_sales.db"
conn = sql.connect("walmart.db")


# Tables from CSV Files
loc = pd.read_csv(store_locations)
trans = pd.read_csv(transactions)
dist = pd.read_csv(distribution_center)
cat = pd.read_csv(product_catalog)

# Tables from database file
inv_levels_db = pd.read_sql("SELECT * FROM inventory_levels;", conn)
daily_sales_db = pd.read_sql("SELECT * FROM daily_sales_summary;", conn)
# Profile on the tables
def profile(df, name=None):
    """Return a summary of dtypes, null counts, null percentages, and unique counts."""
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "null_count": df.isna().sum(),
        "null_pct": (df.isna().mean() * 100).round(2),
        "unique": df.nunique(),
    })
    if name:
        print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns")
    return summary

tables = {"loc": loc, "trans": trans, "dist": dist, "cat": cat, "inv_levels_db":inv_levels_db, "daily_sales_db": daily_sales_db}

for name, df in tables.items():
    display(profile(df, name))
    print()
# Store Location Cleaning
# Normalizing spelling of Florida to FLORIDA
loc["state"] = (
    loc["state"]
    .astype(str)
    .str.upper()
    .str.strip()
    .str.replace(".","", regex=False)
)
state_map = {
    "FLORIDA":"FLORIDA",
    "FLA":"FLORIDA",
    "FL":"FLORIDA",
}
loc["state"]=loc["state"].replace(state_map)

# Fixing store_id to drop duplicates
id_fix = {
    "FL239":"FL-239",
    "FL_287":"FL-287",
}
loc["store_id"]=loc["store_id"].replace(id_fix)
loc = loc.drop_duplicates(subset="store_id", keep="first").reset_index(drop=True)
# Distribution Cleaning

# Removing "~" and converting "surge_capacity" to Int64
dist["surge_capacity"] = (
    pd.to_numeric(dist["surge_capacity"].astype(str).str.replace("~", "", regex=False), errors="coerce")
    .astype("Int64")
)

# Need to make a decision on what to do for missing cost/shipment for DC-BHM
# Product Catalog Cleaning and SKU Mapping

# Cleaning category by normalizing case and consolidating category labels
cat["category"] = (
    cat["category"].str.strip().str.lower()
    .str.title()
)

category_map = {
        "Breakfast Foods": "Breakfast",
        "Emergency Supplies": "Emergency",
        "Alcoholic Beverages": "Alcohol",
        "Groceries": "Grocery",
        "Baby": "Baby Products",
        "Games & Entertainment":"Entertainment",
        "Household Essentials":"Household",
        "Health & Wellness":"Health",
}
cat["category"]=cat["category"].replace(category_map)

# SKU mapping. SKUs will be consolidated based on product name, ignoring all other columns such as cost, supplier and package size
sku_map = {
        'PT-12':'POPTART-STRAW',
        'SKU#459812':'POPTART-STRAW',
        'POPTART-STRAW':'POPTART-STRAW',
        'PT_STRAWBERRY':'POPTART-STRAW',
        'WTR-24':'WATER24PK',
        'WATER24PK':'WATER24PK',
        'WTR_24PK':'WATER24PK',
        'BATT-100': 'BAT-AA4',
        'BAT-AA4':'BAT-AA4',
        'BATT_AA_4PK':'BAT-AA4',
        'FLASH-01': 'LED-FLASH',
        'LED-FLASH': 'LED-FLASH',
        'BEER-24': 'BEER-24',
        'LTBEER24': 'BEER-24',
        'BG-MONO': 'GAME-MONO',
        'MONOPOLY-01': 'GAME-MONO',
        'GAME-MONO': 'GAME-MONO',
        'CARDS-01':'CARDS-01',
        'PLAYCARD-STD':'CARDS-01',
        'STREAM-01':'STREAM-01',
        'ROKU-STD':'STREAM-01',
        'SOUP-CAN':'SOUP',
        'CNSOUP-01':'SOUP',
        'BREAD-WH': 'WHITEBREAD',
        'WHITEBREAD': 'WHITEBREAD',
        'PB-16': "PNUTBTR",
        'PNUTBTR-16': "PNUTBTR",
        'CNDL-EM': "EMERG-CNDL",
        'CANDLE-EMRG': "EMERG-CNDL",
        'FAID-01': 'FIRSTAID-KIT',
        'FIRSTAID-KIT': 'FIRSTAID-KIT',
        'CHRG-PORT': 'CHRGR',
        'PORTCHARGER': 'CHRGR',
        'CHIPS-VP': "CHIPS-VP",
        'CHIPVARIETY': "CHIPS-VP",
        'PTOWEL-6': 'PTOWEL',
        'PAPERTOWEL6': 'PTOWEL',
        'TP-12': 'TOILETPAPER',
        'TOILETPAPER12': 'TOILETPAPER',
        'DIAP-S3':'DIAPERS',
        'DIAPERS-3':'DIAPERS',
        'BFORM-01':'BABYFORM',
        'BABYFORMULA':'BABYFORM',
        'TARP-HD': 'TARP-HD',
        'HDTARP-01': 'TARP-HD',
}
cat["sku"]=cat["sku"].replace(sku_map)

# Normalizing product_name. This also has functionality for the transaction data
groups = {
    "Pop-Tarts Strawberry":   ["pop-tarts strawberry", "Strawberry Pop-Tart", 'Pop Tarts Strawberry',"Pop-Tarts Strawberry",'POPTART-STRAWBERRY', 'Pop Tarts - Strawberry'],
    "Bottled Water 24-Pack":  ["BOTTLED WATER 24PK", "Bottled Water 24-Pack", "Water - 24 Pack"],
    "AA Batteries 4-Pack":    ["BATTERIES AA 4PK", "AA Batteries 4-Pack",'AA Batteries (4-pack)'],
    "LED Flashlight":         ["FLASHLIGHT LED", "LED Flashlight",'Flashlight - LED'],
    "Light Beer 24-Pack":     ["Light Beer (24-pack)", "Light Beer 24-Pack",'Light Beer 24pk','BEER LIGHT 24PK'],
    "Roku Streaming Stick":   ["Roku Streaming Stick", "Streaming Device",'STREAMING STICK'],
    "Monopoly Board Game":    ["Monopoly Board Game", "Board Game - Monopoly",'MONOPOLY'],
    "Playing Cards":          ["Playing Cards", "PLAYING CARDS",'Standard Playing Cards'],
    "Canned Soup":            ["Soup - Canned", "Canned Soup",'CANNED SOUP'],
    "White Bread":            ["WHITE BREAD", "White Bread",'Bread - White'],
    "Peanut Butter":          ["PEANUT BUTTER", "Peanut Butter",'Peanut Butter 16oz'],
    "Emergency Candles":      ["Candles (Emergency)", "Emergency Candles",'CANDLES - EMERGENCY'],
    "First Aid Kit":          ["First-Aid Kit", "First Aid Kit",'FIRST AID KIT'],
    "Portable Phone Charger": ["PORTABLE CHARGER", "Portable Phone Charger",'Phone Charger - Portable'],
    "Chips Variety Pack":     ["Variety Pack - Chips", "Chips Variety Pack",'CHIPS VARIETY PACK'],
    "Paper Towels":           ["Paper Towels",  'PAPER TOWELS 6PK','Paper Towels 6-Roll'],
    "Toilet Paper 12-Roll":   ["TOILET PAPER 12PK", "Toilet Paper 12-Roll",'TP 12 Pack'],
    "Baby Diapers - Size 3":  ["Baby Diapers - Size 3", "DIAPERS SZ 3",'Diapers Size 3'],
    "Baby Formula":           ["Infant Formula", "Baby Formula", 'BABY FORMULA'],
    "Heavy Duty Tarp":        ["Heavy Duty Tarp", "TARP - HEAVY DUTY",'HD Tarp'],
}


name_map = {variant: canon for canon, variants in groups.items() for variant in variants}
cat["product_name"] = cat["product_name"].str.strip().map(name_map)

# Converting package_size to Int64. For the entries with , they were formerly "bundle", which is not clear on how large the package size is.
size = cat["package_size"].str.strip().str.lower().replace({"single": "1", "bundle": pd.NA})
cat["package_size"] = size.str.extract(r"(\d+)", expand=False).astype("Int64")

# Need to decide on what to do with NaN in the "unit_cost" column
# Transaction data cleaning

# Dropping duplicates based on transaction ID
trans = trans.drop_duplicates(subset="transaction_id", keep="first").reset_index(drop=True)

# Cleaning category by normalizing case and consolidating category labels
trans["category"] = (
    trans["category"].str.strip().str.lower()
    .str.title()
)
trans["category"]=trans["category"].replace(category_map)


# SKU map
trans["sku"]=trans["sku"].replace(sku_map)


# product names
trans["product_name"] = trans["product_name"].str.strip().map(name_map)
trans.isna().sum()

# store_id
canonical = ['FL-239','FL-105','FL-330','FL-412','FL-287','FL-156',
             'FL-501','FL-078','FL-619','FL-445','FL-302','FL-710']

def clean_code(code):
    m = re.fullmatch(r"FL[-_]?(\d{3})", str(code).strip().upper())
    return f"FL-{m.group(1)}" if m else None

trans["store_id"] = trans["store_id"].map(clean_code)

# converting unit price to float by dropping $
trans["unit_price"] = (
    pd.to_numeric(trans["unit_price"].astype(str).str.replace("$", "", regex=False), errors="coerce")
    .astype("Float64")
)



# converting timestamp to datetime. Michael proposes that we drop rows with NaN for timestamp.
# The rows are useless because they cannot help identify trends. Its just under 900 rows, so thats a lot of data getting axed.
# Debating whether or not to replace the "timestamp" column with simply "date". Both are in datetime fornmat
ts = trans["timestamp"].str.strip().replace(
    {"": None, "NULL": None, "N/A": None, "NaN": None, "nan": None}
)
formats = [
    "%Y-%m-%d %H:%M:%S",
    "%Y-%m-%dT%H:%M:%S",
    "%m/%d/%Y %H:%M",
    "%d-%b-%Y %H:%M",
    "%m-%d-%Y %H:%M:%S",
]
parsed = pd.Series(pd.NaT, index=trans.index)
for fmt in formats:
    mask = parsed.isna() & ts.notna()
    parsed[mask] = pd.to_datetime(ts[mask], format=fmt, errors="coerce")
trans["timestamp"] = parsed
trans["date"] = trans["timestamp"].dt.date
trans['date'] = pd.to_datetime(trans['date'])

--2026-10-25 22:54:38--  https://raw.githubusercontent.com/Fespinoza1477/walmart-hurricane-project/main/Data/inventory_and_sales.db
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 352256 (344K) [application/octet-stream]
Saving to: ‘walmart.db’

walmart.db          100%[===================>] 344.00K  --.-KB/s    in 0.004s  

2026-10-25 22:54:38 (78.3 MB/s) - ‘walmart.db’ saved [352256/352256]

loc: 14 rows x 8 columns


,dtype,null_count,null_pct,unique
store_id,object,0,0.00,14
store_name,object,0,0.00,12
city,object,0,0.00,12
state,object,0,0.00,3
zip_code,int64,0,0.00,12
latitude,float64,1,7.14,11
longitude,float64,1,7.14,11
region,object,0,0.00,7



trans: 25,562 rows x 8 columns


,dtype,null_count,null_pct,unique
transaction_id,object,0,0.00,25532
store_id,object,0,0.00,60
timestamp,object,886,3.47,23952
sku,object,0,0.00,45
product_name,object,0,0.00,63
category,object,0,0.00,37
quantity,int64,0,0.00,11
unit_price,object,0,0.00,3668



dist: 5 rows x 8 columns


...

,dtype,null_count,null_pct,unique
sku,object,0,0.00,45
product_name,object,0,0.00,38
category,object,0,0.00,27
subcategory,object,13,28.89,19
unit_cost,float64,5,11.11,40
supplier,object,0,0.00,14
package_size,object,0,0.00,15



inv_levels_db: 4,320 rows x 6 columns


,dtype,null_count,null_pct,unique
date,object,0,0.00,30
store_id,object,0,0.00,12
sku,object,0,0.00,49
units_on_hand,int64,0,0.00,196
units_on_order,float64,626,14.49,101
reorder_point,int64,0,0.00,31



daily_sales_db: 3,600 rows x 5 columns


,dtype,null_count,null_pct,unique
date,object,0,0.0,60
store_id,object,0,0.0,12
category,object,0,0.0,10
total_units,float64,0,0.0,285
total_revenue,float64,0,0.0,3552


### 2 — Your decision log

**TODO:** every cleaning decision, what it did, and how many rows it touched. If your teammates disagree with one of these, note the disagreement — that is useful information, not a problem to hide.

In [ ]:
# TODO: print or display your decision log
# Normalizing spelling of Florida to FLORIDA
# Fixing store_id to drop duplicates
# Removing "~" and converting "surge_capacity" to Int64
# Need to make a decision on what to do for missing cost/shipment for DC-BHM
# Product Catalog Cleaning and SKU Mapping
# Cleaning category by normalizing case and consolidating category labels
# SKU mapping. SKUs will be consolidated based on product name, ignoring all other columns such as cost, supplier and package size
# Normalizing product_name. This also has functionality for the transaction data
# Converting package_size to Int64. For the entries with , they were formerly "bundle", which is not clear on how large the package size is.
# Need to decide on what to do with NaN in the "unit_cost" column
# Cleaning category by normalizing case and consolidating category labels
# converting unit price to float by dropping $
# converting timestamp to datetime. Michael proposes that we drop rows with NaN for timestamp.
# The rows are useless because they cannot help identify trends. Its just under 900 rows, so thats a lot of data getting axed.
# Debating whether or not to replace the "timestamp" column with simply "date". Both are in datetime fornmat

### 3 — SKU consolidation progress

**TODO:** how many distinct SKUs did you start with, how many canonical products do you have now, and what is your largest consolidated group? Show the mapping for at least one product family. Pop-Tarts is a good one to start with.

You do not need this finished — you need it started and defensible.

In [ ]:
# TODO

### 4 — One working weather API call

**TODO:** show the call and the parsed DataFrame. It must include the retry logic and a `source` column. Say which timezone the weather timestamps are in and how you will line them up with the sales timestamps. If it is not working, paste the actual error and what you have tried.

In [ ]:
# TODO

### 5 — One chart

**TODO:** daily order volume across your date range. It does not need to be presentation quality yet, but it does need labeled axes and a title.

Look at it before you submit. Does the shape make sense? An early chart is the fastest way to catch a cleaning bug — if your daily volume has a day at zero or a day at ten times everything else, that is usually your pipeline, not the hurricane.

In [ ]:
# TODO

### 6 — Self-check

Fill in what you have and leave the rest. **The required items are the team setup and the charter.** Everything else is progress we want to see, not a pass or fail. The checks are shape checks; they cannot verify your analysis is right, only that you have gotten far enough to have one.

In [ ]:
required = {
    'team_setup_complete': check_team(team_setup, quiet=True),   # from the team setup section
    'charter_committed': True,         # TODO: True once CHARTER.md shows in the git log above
}

progress = {
    'rows_after_cleaning': None,        # an integer
    'decisions_logged': 15,           # how many entries in your log
    'canonical_products': 20,         # distinct products after consolidation
    'weather_api_rows': 3,           # rows returned by your API call
    'notebook_runs_clean': True,       # True after Restart and Run All
}

def is_done(value):
    return not (value is None or value is False
                or (isinstance(value, str) and value.startswith('TODO')))

def check(required, progress):
    print('REQUIRED')
    for label, value in required.items():
        print('  PASS' if is_done(value) else '  FAIL', '-', label, '=', value)
    print('PROGRESS (fill in what you have)')
    for label, value in progress.items():
        print('  done' if is_done(value) else '  not yet', '-', label, '=', value)
    ok = all(is_done(v) for v in required.values())
    print()
    print('required items complete:', ok)
    print('progress items done:', sum(is_done(v) for v in progress.values()), 'of', len(progress))
    return ok

check(required, progress)

REQUIRED
  PASS - team_setup_complete = True
  PASS - charter_committed = True
PROGRESS (fill in what you have)
  not yet - rows_after_cleaning = None
  done - decisions_logged = 15
  done - canonical_products = 20
  done - weather_api_rows = 3
  done - notebook_runs_clean = True

required items complete: True
progress items done: 4 of 5


True

### 7 — Risk and plan

Two questions, answered concretely.

**a)** What is the single most likely thing to go wrong in the next two weeks? Name a specific technical or team problem, not "running out of time."

Not communicating and not being on top of the project within these next two weeks.

**b)** What are you doing about it this week? Name who is doing it and by when.

This week will be mostly to finish all the questions, make and polish the figures and make sure everything is looking good. Who does what and when will be decided when we meet the next time.

### 8 — Where each question stands

**TODO:** mark each of the five questions. These owners must match the question owners table in your charter. Be honest — this is the section I read first.

| Question | Status | Who owns it |
|---|---|---|
| Q1 demand surge |  draft answer | Sam |
| Q2 SKU consolidation | draft answer | Sam|
| Q3 weather correlation | draft answer | Micheal |
| Q4 store and geography | draft answer | Micheal |
| Q5 signal or artifact | Not started | Fabrizzio |

At the end of week 1, "not started" on Q4 and Q5 is expected. "Not started" on all five means come find me on Monday.